In [41]:
import pandas as pd
import sqlite3
import json
import os
from glob import glob
from datetime import datetime
from bs4 import BeautifulSoup as bs
import requests
import re
from thefuzz import process

In [42]:
# cerca i risultati più recenti degli scraper nella cartella ranks
pre2022 = sorted(glob(f'./ranks/pre2022_*.csv'), key=os.path.getmtime, reverse=True)[0]
post2022 = sorted(glob(f'./ranks/post2022_*.csv'), key=os.path.getmtime, reverse=True)[0]

In [43]:
df1 = pd.read_csv(pre2022, low_memory=False)
df2 = pd.read_csv(post2022, low_memory=False)

In [44]:
# common names
# html = requests.get("https://didattica.polito.it/pls/portal30/preimma.pkg_grad.ing_sep?p_a_acc=2024").text
# soup = bs(html, 'html.parser')
# for a in soup.select("table a"):
#     code = a['href'].split("p_mnemo=")[1]
#     name = a.text
#     pname = re.sub(r" \(.*", "", name)
#     print(f"{code}\t{name}\t{pname}")

In [45]:
df = pd.concat([df1,df2], ignore_index=True)
print(len(df))
# filtro, graduatoria polito con errori
df = df[~((df['totale'] == 0) & (df['esito'] != 'NON AMMESSO'))]
print(len(df))

752223
752069


In [46]:
# read course equivalences
df_names = pd.read_csv('./course-equivalence.csv', sep=";")
modern_name = {}
modern_name_list = {}

for i, row in df_names.iterrows():
    if isinstance(row['name'], str) and len(row['name']) > 3:
        modern_name_list[row['code']] = row['name']
        modern_name[row['name']] = row['code']
        for j in range(1,6):
            alt_name = row[f'name{j}']
            if isinstance(alt_name, str) and len(alt_name) > 3:
                modern_name[alt_name] = row['code']
            
with open('datasets/courses.json', 'w') as outfile:
    json.dump(modern_name_list, outfile)

In [47]:
# df.to_csv("test.csv", sep=";")

In [48]:
def rimpiazza_nome(old_name):
    if old_name == "Ingegneria":
        return old_name
    if not isinstance(old_name, str):
        return old_name
    return modern_name[old_name]

# replace course specif
# rimpiazza i nomi dei corsi "strani" con un nome canonico
for c in ["course", "corso di immatricolazione", "corso unico", "corso assegnato", "corso prenotato"]:
    df[c] = df[c].apply(lambda x: rimpiazza_nome(x))
    # df[c] = df[c].str.replace(" \(.+","", regex=True).str.strip()

# set superset uppercase for consistency with other courses
# i supercorsi sono DESIGN, INGEGNERIA, ARCHITETTURA, PIANIFICAZIONE
df['supercorso'] = df['course'].str.upper()
del df['course']

In [49]:
c

'corso prenotato'

In [50]:
# esporto il tutto in formato sqlite, idoneo a fare estrazioni
sqlitefile = f"out/tilstat_{str(datetime.now()).replace(':','-')}.sqlite"
conn = sqlite3.connect(sqlitefile)
df.to_sql('tilstat', conn, if_exists='replace', index=False)

752069

In [51]:
sqlitefile = sorted(glob(f'./out/tilstat_*.sqlite'), key=os.path.getmtime, reverse=True)[0]
conn = sqlite3.connect(sqlitefile)

In [52]:
# PARANOID CHECKS (ogni riga ha almeno un corso tra imma, unico e assegnato)

cur = conn.cursor()
cur.execute("""
    SELECT
        ("corso di immatricolazione" IS NOT NULL +
         "corso unico" IS NOT NULL +
         "corso assegnato" IS NOT NULL) as chk,
         count(*)
    FROM tilstat
    GROUP BY
        chk 
""")
assert cur.fetchall() == [(1,len(df))]

In [53]:
# TIPOLOGIE ESITO

cur = conn.cursor()
cur.execute("""
    SELECT
        distinct(esito)
    FROM tilstat
    
""")
#list(map(lambda x: x[0], cur.description)),
cur.fetchall()

[('ASSEGNATO',),
 ('PRENOTATO',),
 ('LISTA DI ATTESA',),
 ('NON AMMESSO',),
 ("RINUNCIA ALL'IMMATRICOLAZIONE",),
 ('IMMATRICOLATO',),
 ('RINUNCIA ALLA LISTA DI ATTESA',),
 ('ASSEGNATO (**)',),
 ('PRENOTATO (**)',),
 (None,),
 ('RINUNCIA DOPO IMMA',),
 ('NON AMMESSO  - IMMATRICOLAZIONI CHIUSE AI SENSI DELL’ART 9 PUNTO 4 del D.R. N. 121/2018',),
 ('NON AMMESSO - IMMATRICOLAZIONI CHIUSE AI SENSI DELL’ART 9 PUNTO 4 del D.R. N. 121/2018',),
 ('NON AMMESSO  - IMMATRICOLAZIONI CHIUSE AI SENSI DELL’ART 8 PUNTO 6 del D.R. N. 122/2018',),
 ('NON AMMESSO - IMMATRICOLAZIONI CHIUSE AI SENSI DEL D.R. N. 1112/2019',),
 ('AMMESSA AD ANNI SUCCESSIVI AL PRIMO',),
 ('NON AMMESSO - GRADUATORIE CHIUSE',),
 ('NON AMMESSO: REQUISITI LINGUISTICI NON RISPETTATI',),
 ('RINUNICA DOPO IMMA',),
 ('ANNULLAMENTO IMMATRICOLAZIONE',),
 ('IMMATRICOLAZIONE ANNULLATA',)]

In [54]:
# ANNI

cur = conn.cursor()
cur.execute("""
    SELECT DISTINCT year FROM tilstat
    
""")
#list(map(lambda x: x[0], cur.description)),
print(cur.fetchall())

cur = conn.cursor()
cur.execute("""
    SELECT DISTINCT year FROM (SELECT
        CAST(substr(date,1,4) as INTEGER) as year
    FROM tilstat)
    
""")
#list(map(lambda x: x[0], cur.description)),
print(cur.fetchall())

[(2018,), (2019,), (2020,), (2021,), (2022,), (2023,), (2024,)]
[(2017,), (2018,), (2019,), (2020,), (2021,), (2022,), (2023,)]


In [55]:
# TILSTAT

cur = conn.cursor()

cur.execute(f"""
    SELECT
        COALESCE("corso unico", "corso assegnato", "corso prenotato", "supercorso") as corso,
        
        /* grad_n <=> specific date in a year */
        CAST(substr(date,9,2) as INTEGER) / 8 as week,
        substr(date,6,2) as month,
        year,
        
        MIN(grad_n) as first_scorr,
        MAX(grad_n) as last_scorr,
        COUNT(*) as people_n,
        MIN(totale) as min_tot,
        
        COALESCE("user", "af user") as matr,
        
        /* following star is a sqlite magic: can return all line of MIN value, to debug and stats purpose */ 
        *
    FROM
        tilstat as ts1
    WHERE
        /* select only taken students */
        ("esito" LIKE "ASSEGNATO" OR
        "esito" LIKE "PRENOTATO" OR
        "esito" LIKE "IMMATRICOLATO") AND
        
        /* contig or not */
        conting = 0 AND
        
        instr(matr, "*") = 0
    GROUP BY
        corso, year, month, week
    HAVING
        /* hide super-corso of alredy shown course */
        corso <> "INGEGNERIA"
    
    /* this order prevent inversions in view */
    ORDER BY
        month, week
    
""")
headers = list(map(lambda x: x[0], cur.description))
out = [{h:v for h,v in zip(headers, row)} for row in cur.fetchall()]
with open('datasets/tilstat.json', 'w') as outfile:
    json.dump(out, outfile)

In [56]:
# TILSTAT

cur = conn.cursor()

cur.execute(f"""
    SELECT corso, year, COUNT(distinct matr) as cnt FROM
    
    (SELECT
        year,
        COALESCE("corso unico", "corso assegnato", "corso prenotato", "supercorso") as corso,
        COALESCE("user", "af user") as matr
    FROM
        tilstat as ts1)
        
    GROUP BY corso, year
    HAVING
        /* hide super-corso of alredy shown course */
    corso <> "INGEGNERIA"
""")
headers = list(map(lambda x: x[0], cur.description))
out = [{h:v for h,v in zip(headers, row)} for row in cur.fetchall()]
for row in out:
    print(modern_name_list[row['corso']],"\t", row['year'], "\t", row['cnt'])

INGEGNERIA AEROSPAZIALE (L-9) 	 2018 	 116
INGEGNERIA AEROSPAZIALE (L-9) 	 2019 	 181
INGEGNERIA AEROSPAZIALE (L-9) 	 2020 	 191
INGEGNERIA AEROSPAZIALE (L-9) 	 2021 	 150
INGEGNERIA AEROSPAZIALE (L-9) 	 2022 	 1489
INGEGNERIA AEROSPAZIALE (L-9) 	 2023 	 1269
INGEGNERIA AEROSPAZIALE (L-9) 	 2024 	 1231
INGEGNERIA PER L'AMBIENTE E IL TERRITORIO (L-7) 	 2018 	 144
INGEGNERIA PER L'AMBIENTE E IL TERRITORIO (L-7) 	 2019 	 228
INGEGNERIA PER L'AMBIENTE E IL TERRITORIO (L-7) 	 2020 	 170
INGEGNERIA PER L'AMBIENTE E IL TERRITORIO (L-7) 	 2021 	 212
INGEGNERIA PER L'AMBIENTE E IL TERRITORIO (L-7) 	 2022 	 396
INGEGNERIA PER L'AMBIENTE E IL TERRITORIO (L-7) 	 2023 	 287
INGEGNERIA PER L'AMBIENTE E IL TERRITORIO (L-7) 	 2024 	 307
ARCHITETTURA (ARCHITECTURE) 	 2021 	 674
ARCHITETTURA (ARCHITECTURE) 	 2022 	 771
ARCHITETTURA (ARCHITECTURE) 	 2023 	 1096
ARCHITETTURA (ARCHITECTURE) 	 2024 	 1306
INGEGNERIA DELL'AUTOVEICOLO (L-9); INGEGNERIA DELL'AUTOVEICOLO (AUTOMOTIVE ENGINEERING) (L-9) 	 2018 	 

In [57]:
# TILSTAT

cur = conn.cursor()

cur.execute(f"""
    SELECT year, COUNT(distinct matr) as cnt FROM
    
    (SELECT
        year,
        COALESCE("user", "af user") as matr
    FROM
        tilstat as ts1)
        
    GROUP BY year
""")
headers = list(map(lambda x: x[0], cur.description))
out = [{h:v for h,v in zip(headers, row)} for row in cur.fetchall()]
out

[{'year': 2018, 'cnt': 5573},
 {'year': 2019, 'cnt': 9372},
 {'year': 2020, 'cnt': 9601},
 {'year': 2021, 'cnt': 8995},
 {'year': 2022, 'cnt': 8916},
 {'year': 2023, 'cnt': 8975},
 {'year': 2024, 'cnt': 9051}]

In [58]:
out

[{'year': 2018, 'cnt': 5573},
 {'year': 2019, 'cnt': 9372},
 {'year': 2020, 'cnt': 9601},
 {'year': 2021, 'cnt': 8995},
 {'year': 2022, 'cnt': 8916},
 {'year': 2023, 'cnt': 8975},
 {'year': 2024, 'cnt': 9051}]

In [59]:
# TILSTAT

cur = conn.cursor()

cur.execute(f"""
    SELECT
        DISTINCT( COALESCE(
            "corso unico",
            "corso assegnato",
            "corso prenotato",
            "corso di immatricolazione")) as corso
    FROM tilstat
    
""")
headers = list(map(lambda x: x[0], cur.description))
courses = [c[0] for c in cur.fetchall() if c[0] is not None]
courses


['AMB1T1',
 'MTM1T1',
 'INF1T3',
 'BIO1T1',
 'GES1T4',
 'AER1T1',
 'ELN1T3',
 'CHI1T1',
 'MEC1T1',
 'PRO1B1',
 'EDI1T1',
 'CIN1T3',
 'MAT1T1',
 'ENE1T1',
 'AUT1T1',
 'ECE1T3',
 'FIS1T3',
 'PRO1A1',
 'PRO1N1',
 'ELT1T1',
 'CIV1T1',
 'ARCHITETTURA',
 'CEE1T1']

In [60]:
# TILSTAT HOF

out = []

for c in courses:
    cur = conn.cursor()
    cur.execute(f"""
        SELECT
            COALESCE(
                "corso/i in lista di attesa",
                "corsi in lista di attesa",
                "corso unico",
                "corso assegnato",
                "corso prenotato",
                "supercorso"
            ) as corso,
            COALESCE("user", "af user") as matr,
            year,
            COUNT(*) as count
        FROM
            tilstat
        WHERE      
            /* contig or not */
            conting = 0 AND

            /* remove special matr */
            instr(matr, "*") = 0 AND

            grad_n = 0 AND
            corso LIKE "%{c}%"

        GROUP BY
            year

    """)
    headers = list(map(lambda x: x[0], cur.description))
    out += [{h:v for h,v in zip(headers, row)} for row in cur.fetchall()]
    
with open('dataset/tilstathof.json', 'w') as outfile:
    json.dump(out, outfile)
out

FileNotFoundError: [Errno 2] No such file or directory: 'dataset/tilstathof.json'

In [ ]:
# COMPLESSITA'

cur = conn.cursor()
cur.execute(f"""
        
    SELECT
        year,
        count(*) as cnt,
        avg(mt) as avg,
        AVG(mt*mt) - AVG(mt)*AVG(mt) as variance
    FROM
        (
        SELECT
            year,
            COALESCE("user", "af user") as matr,
            COALESCE(
                "corso di immatricolazione",
                "corso unico",
                "corso assegnato",
                "corso prenotato",
                "supercorso"
            ) as corso,
            
            /*
                This aggregation is arbitrary. For students applies only one rank the
                results is the same with evrey aggregation. For other, i choose simply take the maximum.
                With avg the results wont change a lot.
            */
            MAX(totale) as mt 
        FROM
            tilstat
        WHERE
            esito LIKE "%IMMATRICOLATO%"
            AND corso LIKE "%AEROSPAZIALE%"
            /*esito NOT LIKE "%RINUNCIA%" and esito NOT LIKE "%ATTESA%"*/
        GROUP BY 
            year, matr
        )
    GROUP BY 
        year

""")
headers = list(map(lambda x: x[0], cur.description))
out = [{h:v for h,v in zip(headers, row)} for row in cur.fetchall()]

# with open('tilstatquart.json', 'w') as outfile:
#     json.dump(out, outfile)
out[:100]

In [ ]:
# {'year': 2022,
#   'cnt': 9227,
#   'avg': 48.51357149920639,
#   'variance': 348.15130310044333}]

In [ ]:
# COMPLESSITA'

cur = conn.cursor()
cur.execute(f"""
        
    SELECT
        year,
        count(*) as cnt,
        avg(totale) as mean,
        (AVG(totale*totale) - AVG(totale)*AVG(totale)) as std,
        (esito LIKE "%IMMATRICOLATO%") as preso
    FROM
    
        (
        SELECT
           COALESCE(
                "corso di immatricolazione",
                "corso unico",
                "corso assegnato",
                "corso prenotato",
                "supercorso"
            ) as corso,
            COALESCE("user", "af user") as matr, year, uri,totale, esito,
            max(grad_n)
        FROM
            tilstat
        WHERE      
           1
        GROUP BY 
            year, corso, matr
        HAVING
            esito NOT LIKE "%RINUNCIA" or 1
        )
    
    group by year
    /*GROUP BY year , preso
    having preso = 1*/

""")
headers = list(map(lambda x: x[0], cur.description))
out = [{h:v for h,v in zip(headers, row)} for row in cur.fetchall()]

# with open('tilstatquart.json', 'w') as outfile:
#     json.dump(out, outfile)
out[:100]

In [ ]:
# # PARANOID CHECKS

# cur = conn.cursor()
# cur.execute("""
#     SELECT
#         date, uri, count(*)
#     FROM tilstat
#     WHERE
#         (
#          "course" == "Ingegneria" AND
         
#          "corso di immatricolazione" IS NULL AND
#          "corso unico" IS NULL AND
#          "corso assegnato" IS NULL AND
#          "corso prenotato" IS NULL AND
#          "corsi in lista di attesa" IS NULL AND
         
#          "esito" NOT LIKE "NON AMMESSO%" AND
#          "esito" NOT LIKE "RINUNCIA ALL'IMMATRICOLAZIONE%" AND
#          "esito" NOT LIKE "RINUNCIA ALLA LISTA DI ATTESA%" AND 
#          "esito" NOT LIKE "RINUNCIA DOPO IMMA%"
#          )
#     GROUP BY date
    
# """)
# list(map(lambda x: x[0], cur.description)), cur.fetchall()

In [ ]:
cur = conn.cursor()
cur.execute("""
    SELECT * FROM tilstat 
""")
list(map(lambda x: x[0], cur.description))

In [ ]:
#















































